# 02 · S&P 500 index effect: event study, calendar-time portfolios, and costs

**Research question.** Has the price effect of S&P 500 additions and deletions disappeared, and does a version of it survive transaction costs?

The notebook runs the full sample definition from `01_inventory.ipynb`. It needs no prepared files and writes nothing to disk. It has four parts.

1. **Event study.** Market-adjusted cumulative abnormal returns (CARs) for additions and deletions, by sub-period.
2. **Calendar-time portfolios.** Long deletions, short additions, and the long–short spread. Alphas use the Fama–French five factors plus momentum.
3. **Costs.** Net alphas after spread-based or assumed trading costs and short-borrow fees.
4. **Robustness.** Value weighting, holding periods, influential events, the delisting-return assumption, and the provisional 2023–2025 period.

**Samples.** The core sample is 1976–2022, where the membership data are complete. Events from 2023–2025 rely on manually collected, provisional additions and removals. They appear only in rows labelled as a robustness check.

**Main event samples.** Additions exclude securities without CRSP history in the three months before the event, a proxy for spin-offs and merger successors. Deletions exclude securities that delist from CRSP before, or within 21 sessions after, the deletion. Those are mergers, acquisitions, and failures rather than pure index deletions. Both flags use only information available at the event date (see notebook 01).

**Announcement dates.** No verified announcement date exists in the core sample. Every window is therefore centred on the index trading date and not on the announcement. The pre-event window mixes announcement returns with anticipation. A rise during [−60, −1] cannot be attributed to either mechanism separately.

All specifications below were fixed before any result was computed. The final section counts them.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd
import statsmodels.api as sm

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"
with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

sub_periods = ["1976–1989", "1990–1999", "2000–2009", "2010–2022"]
overall_period = "1976–2022"
all_periods = [overall_period, *sub_periods]


def period_of(years):
    """Sub-period label for calendar years in the core sample, missing outside it."""
    return pd.cut(years, bins=[1975, 1989, 1999, 2009, 2022], labels=sub_periods).astype(object)


# Every reported estimate (mean CAR, difference test, or alpha) is logged to count specifications.
specification_log = []


def log_specifications(table_name, table):
    """Record the t-statistics of a results table (one per reported estimate)."""
    t_columns = [column for column in table.columns
                 if (column[-1] if isinstance(column, tuple) else column) == "t_stat"]
    t_values = pd.concat([table[column] for column in t_columns]).dropna()
    specification_log.append({
        "table": table_name, "estimates": len(t_values),
        "|t| > 2": int(t_values.abs().gt(2).sum()), "|t| > 3": int(t_values.abs().gt(3).sum()),
    })


main_additions = additions_register.loc[additions_register["main_sample"]]
main_deletions = deletions_register.loc[deletions_register["main_sample"]]
recent_additions_sample = additions_register.loc[
    additions_register["sample"].eq("robustness_2023_2025") & additions_register["permno"].notna()
    & ~additions_register["flag_no_prior_crsp_3m"]
]
recent_deletions_sample = deletions_register.loc[
    deletions_register["sample"].eq("robustness_2023_2025") & deletions_register["permno"].notna()
    & ~deletions_register["flag_delisting_related"]
]
display(pd.DataFrame({
    "additions": [len(main_additions), len(recent_additions_sample)],
    "deletions": [len(main_deletions), len(recent_deletions_sample)],
}, index=["Main sample, 1976–2022", "Robustness, 2023–2025 (provisional)"]))

## Daily CRSP data

The daily file is read from the Parquet copy in `Data/parquet/crsp_daily` when it is available. The cell checks that the copy has the same number of rows as the daily CSV export, 92,586,611 excluding the header. A 200,000-row sample matched the CSV value for value on returns, prices, quotes, volume, and capitalization. Without the Parquet copy, the cell scans the CSV in chunks, which takes considerably longer.

Only securities in the event registers are kept. Identical duplicate rows are collapsed. When rows for the same security and date disagree, returns and quotes for that date are set to missing.

CRSP records a delisting return as an extra daily row on the session after the last trade, with `DlyDelFlg = "Y"`. That row is the last return a holder receives. If its return is missing, the abnormal return on that row is set to zero, which is equivalent to assuming the proceeds earn the S&P 500 return. After delisting, the abnormal return is zero.

In [ ]:
expected_daily_rows = 92_586_611  # data rows in the daily CSV export (line count minus header), checked 2026-11-07
daily_fields = {
    "permno": "PERMNO", "dlycaldt": "DlyCalDt", "dlydelflg": "DlyDelFlg", "dlyprc": "DlyPrc",
    "dlyret": "DlyRet", "dlyretx": "DlyRetx", "dlybid": "DlyBid", "dlyask": "DlyAsk",
}
event_permnos = sorted(set(
    pd.concat([additions_register["permno"], deletions_register["permno"]]).dropna().astype(int)
))
parquet_dir = data_dir / "parquet" / "crsp_daily"
if parquet_dir.is_dir():
    import pyarrow.dataset as ds

    daily_dataset = ds.dataset(parquet_dir, format="parquet")
    if daily_dataset.count_rows() != expected_daily_rows:
        raise ValueError("The Parquet copy of the daily file does not match the CSV row count.")
    daily = daily_dataset.to_table(
        columns=list(daily_fields), filter=ds.field("permno").isin(event_permnos)
    ).to_pandas()
else:
    daily = pd.concat(
        chunk.loc[chunk["PERMNO"].isin(event_permnos)]
        for chunk in pd.read_csv(paths["crsp_daily"], usecols=list(daily_fields.values()), chunksize=1_000_000)
    ).rename(columns={csv_name: name for name, csv_name in daily_fields.items()})

daily["date"] = pd.to_datetime(daily.pop("dlycaldt"))
rows_read = len(daily)
daily = daily.drop_duplicates()
conflicting = daily.duplicated(["permno", "date"], keep=False)
daily.loc[conflicting, ["dlyret", "dlyretx", "dlybid", "dlyask"]] = np.nan
daily = daily.drop_duplicates(["permno", "date"]).copy()
daily["session"] = daily["date"].map(session_of)
off_calendar = int(daily["session"].isna().sum())
daily = daily.dropna(subset=["session"]).astype({"session": "int64"})

daily_benchmark = pd.read_csv(
    paths["index_daily"], usecols=["dlycaldt", "sprtrn"], parse_dates=["dlycaldt"]
).rename(columns={"dlycaldt": "date", "sprtrn": "sp500_return"})
daily = daily.merge(daily_benchmark, on="date", how="left", validate="many_to_one")
delisting_row = daily["dlydelflg"].eq("Y")
daily["ar"] = daily["dlyret"] - daily["sp500_return"]
daily["ar_price"] = daily["dlyretx"] - daily["sp500_return"]
daily.loc[delisting_row & daily["dlyret"].isna(), "ar"] = 0.0
daily.loc[delisting_row & daily["dlyretx"].isna(), "ar_price"] = 0.0
daily = daily.sort_values(["permno", "session"], ignore_index=True)
permno_span = daily.groupby("permno")["session"].agg(first_session="min", last_session="max")
last_calendar_session = len(trading_calendar) - 1

display(pd.Series({
    "Rows read for event securities": rows_read,
    "Rows after collapsing identical duplicates": len(daily) + off_calendar,
    "Security-dates with conflicting rows (set to missing)": int(conflicting.sum() // 2),
    "Rows dated off the trading calendar (dropped)": off_calendar,
    "Delisting-return rows": int(delisting_row.sum()),
    "  of which the delisting return is missing (AR set to 0)": int((delisting_row & daily["dlyret"].isna()).sum()),
}, name="daily data"))

## Event study: design

**Event day 0** is the index trading date, the session whose close index funds use to rebalance. For an addition, this is the session before the effective date, because S&P adds stocks before the open of the effective date. For a deletion, it is the last session in the index. Centring both event types on the same trading close makes the windows comparable. A robustness check centres additions on the effective date itself. For deletions the two dates coincide.

**Abnormal return.** The daily abnormal return is the CRSP total return minus the S&P 500 return (`sprtrn`, a price index). The benchmark excludes dividends, so the CAR has an upward bias of roughly the stock's daily dividend yield. That bias is negligible in short windows but is about 1–3% over 250 sessions. A robustness check therefore uses price returns for both the stock and the index.

**CAR and windows.** The CAR is the sum of daily abnormal returns over [−60, −1], [0, +20], [+1, +60], and [+1, +250]. An event enters a window only if it has a return for every session it is listed in that window, the calendar covers the window, and, for a window that starts before day 0, the security trades before the window starts. These conditions do not depend on returns after the event.

**Inference.** The t-statistic tests a zero mean CAR across events, with standard errors clustered by the calendar month of the effective date. Events in the same month share index rebalances and market shocks. Sub-periods are defined by event year. The change between 1990–1999 and 2010–2022 is the coefficient on a 2010–2022 indicator in a regression of the CAR on a constant, using events from those two decades and the same clustering.

Because effective dates before about 1990 lie on a weekly grid (notebook 01), the short windows in 1976–1989 are measured with error.

In [ ]:
event_windows = {"[-60,-1]": (-60, -1), "[0,+20]": (0, 20), "[+1,+60]": (1, 60), "[+1,+250]": (1, 250)}


def study_events(register):
    events = register[["permno", "effective_date", "index_trade_date"]].copy()
    events["permno"] = events["permno"].astype(int)
    events["event_id"] = np.arange(len(events))
    events["cluster_month"] = events["effective_date"].dt.to_period("M")
    events["period"] = period_of(events["effective_date"].dt.year)
    return events.set_index("event_id")


def compute_cars(events, anchor_column, ar_column="ar"):
    """CAR per event and window, with a flag for complete windows."""
    anchors = events[["permno"]].assign(anchor=events[anchor_column].map(session_of)).join(permno_span, on="permno")
    if anchors["anchor"].isna().any():
        raise ValueError("An event date is not a trading session.")
    rows = anchors.reset_index()[["event_id", "permno", "anchor"]].merge(
        daily[["permno", "session", ar_column]], on="permno"
    )
    rows["relative_session"] = rows["session"] - rows["anchor"]
    rows = rows.loc[rows["relative_session"].between(-60, 250)]
    results = []
    for window, (low, high) in event_windows.items():
        in_window = rows.loc[rows["relative_session"].between(low, high)]
        sums = in_window.groupby("event_id")[ar_column].agg(car="sum", observed="count")
        frame = anchors.join(sums)
        start, end = frame["anchor"] + low, frame["anchor"] + high
        expected = (np.minimum(end, frame["last_session"]) - start + 1).clip(lower=0)
        frame["complete"] = (
            frame["first_session"].lt(start) & end.le(last_calendar_session)
            & expected.gt(0) & frame["observed"].fillna(0).eq(expected)
        )
        results.append(frame[["car", "complete"]].assign(window=window))
    return pd.concat(results).join(events[["cluster_month", "period"]])


def clustered_mean(values, clusters):
    groups = pd.factorize(clusters)[0]
    if len(values) < 3 or len(np.unique(groups)) < 2:
        return np.nan
    fit = sm.OLS(np.asarray(values, float), np.ones(len(values))).fit(
        cov_type="cluster", cov_kwds={"groups": groups}
    )
    return fit.tvalues[0]


def summarize_cars(cars, periods=all_periods):
    rows = []
    for window in event_windows:
        usable = cars.loc[cars["window"].eq(window) & cars["complete"]]
        for period in periods:
            sample = usable if period == overall_period else usable.loc[usable["period"].eq(period)]
            rows.append({
                "window": window, "period": period, "events": len(sample),
                "mean_car_pct": 100 * sample["car"].mean(),
                "t_stat": clustered_mean(sample["car"], sample["cluster_month"]),
                "median_car_pct": 100 * sample["car"].median(),
                "share_positive": sample["car"].gt(0).mean(),
            })
    return pd.DataFrame(rows).set_index(["window", "period"])


def difference_test(cars, early="1990–1999", late="2010–2022"):
    rows = []
    for window in event_windows:
        sample = cars.loc[cars["window"].eq(window) & cars["complete"] & cars["period"].isin([early, late])]
        is_late = sample["period"].eq(late).astype(float).to_numpy()
        fit = sm.OLS(sample["car"].to_numpy(), np.column_stack([np.ones(len(sample)), is_late])).fit(
            cov_type="cluster", cov_kwds={"groups": pd.factorize(sample["cluster_month"])[0]}
        )
        rows.append({
            "window": window, f"events {early}": int((1 - is_late).sum()), f"events {late}": int(is_late.sum()),
            "difference_pct": 100 * fit.params[1], "t_stat": fit.tvalues[1],
        })
    return pd.DataFrame(rows).set_index("window")


addition_events = study_events(main_additions)
deletion_events = study_events(main_deletions)
addition_cars = compute_cars(addition_events, "index_trade_date")
deletion_cars = compute_cars(deletion_events, "index_trade_date")

coverage = pd.DataFrame({
    "additions: complete": addition_cars.groupby("window")["complete"].sum(),
    "additions: events": addition_cars.groupby("window").size(),
    "deletions: complete": deletion_cars.groupby("window")["complete"].sum(),
    "deletions: events": deletion_cars.groupby("window").size(),
}).loc[list(event_windows)]
display(coverage)

## Event study: main results

Mean CAR in percent, with t-statistics clustered by effective month. Windows are centred on the index trading date (day 0). Additions are expected to show positive CARs up to day 0 and a reversal afterwards. Deletions are expected to show the opposite pattern.

In [ ]:
addition_summary = summarize_cars(addition_cars)
deletion_summary = summarize_cars(deletion_cars)
event_study_main = pd.concat({"additions": addition_summary, "deletions": deletion_summary}, axis=1)
display(event_study_main.round(3))
log_specifications("Event study, main", event_study_main)

print("Change in mean CAR, 2010–2022 minus 1990–1999 (percentage points, clustered t-statistic)")
event_study_difference = pd.concat(
    {"additions": difference_test(addition_cars), "deletions": difference_test(deletion_cars)}, axis=0
)
display(event_study_difference.round(3))
log_specifications("Event study, 2010–2022 vs 1990–1999 difference", event_study_difference)

### Interpretation of the event study

The figures below come from the run of 7 November 2026. The tables above are authoritative if the data change.

**Additions.** The pre-event run-up is large and has not disappeared. The mean CAR over [−60, −1] is 7.3% for 1976–2022 (t = 10.4) and 9.0% for 2010–2022 (t = 8.7), against 8.4% in 1990–1999. The run-up does not measure the index effect alone. It mixes announcement returns, anticipation, and selection: S&P adds firms whose market value has grown, so stocks with strong past returns are over-represented. After the index trading date, 1990–1999 shows a reversal of −4.2% over [+1, +60] (t = −2.8). In 2010–2022 there is no reversal (+0.2%, t = 0.2), and the change from the 1990s is +4.3 percentage points (t = 2.5). Over [+1, +250], no sub-period after 1989 differs reliably from zero.

**Deletions.** Deleted stocks fall before the event: −13.4% over [−60, −1] (t = −4.9), and −10.7% in 2010–2022 (t = −4.8). They then rebound strongly in 1990–2009: +12.8% (t = 3.3) and +27.4% (t = 4.1) over [+1, +60], and +21.8% and +42.4% over [+1, +250]. The means in 2000–2009 are dominated by a few distressed stocks that recovered after 2001 and 2008 (Fannie Mae, Freddie Mac, AMR, and others). The medians are smaller but still large: +12.9% over [+1, +60] and +28.4% over [+1, +250]. In 2010–2022 the rebound is absent: +0.6% over [+1, +60] (t = 0.2) and +0.2% over [+1, +250] (t = 0.04). The decline from the 1990s is significant: −12.3 percentage points (t = −2.6) and −21.6 percentage points (t = −2.2).

**1976–1989.** This period behaves differently. Additions keep rising after the event (+4.3% over [+1, +60], t = 4.8), and deletions show no pre-event decline. Effective dates in this period lie on a weekly grid and S&P did not pre-announce changes before October 1989. Both facts make this period a poor benchmark for the post-event windows.

Because the windows are centred on the index trading date and announcement dates are missing, the size of the announcement-day effect itself cannot be measured here.

## Event study: robustness

1. **Additions centred on the effective date.** Day 0 is the effective date instead of the session before it. The deletion results are unchanged by construction, because the two dates coincide for deletions.
2. **Price returns.** Stock price return (`DlyRetx`) minus the S&P 500 price return, so neither side includes dividends. This is reported for the full core sample.
3. **2023–2025 (provisional).** The same design applied to the provisional events. Windows that run past December 2025 are incomplete, so the long windows contain few events.

In [ ]:
addition_cars_effective = compute_cars(addition_events, "effective_date")
robust_effective = summarize_cars(addition_cars_effective)
print("Additions centred on the effective date")
display(robust_effective.round(3))
log_specifications("Event study, additions centred on effective date", robust_effective)

robust_price = pd.concat({
    "additions": summarize_cars(compute_cars(addition_events, "index_trade_date", "ar_price"), [overall_period]),
    "deletions": summarize_cars(compute_cars(deletion_events, "index_trade_date", "ar_price"), [overall_period]),
}, axis=1)
print("Price-return CARs, 1976–2022")
display(robust_price.round(3))
log_specifications("Event study, price-return CARs", robust_price)

recent_label = "2023–2025 (provisional)"
recent_event_frames = {"additions": study_events(recent_additions_sample), "deletions": study_events(recent_deletions_sample)}
robust_recent = {}
for name, events in recent_event_frames.items():
    cars = compute_cars(events, "index_trade_date")
    cars["period"] = recent_label
    robust_recent[name] = summarize_cars(cars, [recent_label]).droplevel("period")
robust_recent = pd.concat(robust_recent, axis=1)
print("Robustness check: 2023–2025 events (provisional membership)")
display(robust_recent.round(3))
log_specifications("Event study, 2023–2025 provisional", robust_recent)

### Interpretation of the event-study robustness checks

Centring additions on the effective date instead of the index trading date moves the trading-day return into the pre-event window. In 1990–1999 the [0, +20] CAR becomes −3.7% (t = −5.1) instead of −0.7%. The effect of index-fund demand is therefore concentrated in the close of the trading date and reverses on the next sessions. In 2010–2022 the [0, +20] CAR stays close to zero under either centring (+0.3% and +0.4%). Price-return CARs are slightly smaller than total-return CARs, as expected from the dividend mismatch, and do not change any conclusion.

The 2023–2025 events show a strong run-up for additions (+23.6% over [−60, −1], t = 4.3; 43 events), a short-run reversal (−4.5% over [0, +20], t = −2.1), and a pre-event decline for deletions (−15.6%, t = −3.6). Deletions do not rebound in this period. These events are few, membership is provisional, and the long windows are incomplete for 2025 events, so this period is a check only.

## Calendar-time portfolios: design

Each event enters its portfolio at the start of the month after its **effective month** and is held for 12 months. A March event is held from April through the following March. The effective date and the index trading date both fall no later than the end of the effective month, so the position never uses that month's closing prices. Each month the portfolio holds every event whose holding period covers that month.

- **Long deletions:** main-sample deletions, whose stocks keep trading.
- **Short additions:** main-sample additions. The leg earns the risk-free rate on the short proceeds minus the additions' return, so its return is `RF − R_additions`.
- **Long–short spread:** `R_deletions − R_additions`. This is a zero-investment return and is regressed without subtracting the risk-free rate.

Portfolios are equal weighted and rebalanced monthly. The value-weighted check uses market capitalisation at the end of the previous month. If a security re-enters the same leg during its holding period, it is held once, under the more recent event.

**Returns and delisting.** Monthly returns are CRSP `MthRet`, which includes the delisting return in the delisting month. A position ends after its delisting month. For the flags `G`, `M`, and `V`, CRSP reports a missing or partial delisting return, so the delisting month carries only the partial-month return. The main specification uses that return as reported. The sensitivity check compounds an assumed −30% delisting return for those cases unless the delisting is a merger or exchange (`MER`, `GEX`). The −30% follows the usual assumption for performance-related delistings. A position-month without any CRSP return is left out of that month's average, and the number of such cases is reported.

**Alpha.** Monthly returns are regressed on the Fama–French five factors and momentum. Standard errors are Newey–West with 12 lags, using the estimation code from the earlier post-inclusion prototype. A month in which a leg holds no position is left blank but keeps its calendar position. Sub-periods and the 1976–2022 core sample are defined by return month. Only core-sample events enter the core portfolios.

In [ ]:
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]
monthly_factors = read_monthly_factors(
    data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
).join(read_monthly_factors(data_dir / "F-F_Momentum_Factor_CSV.zip", ["Mom"]), validate="one_to_one")


def estimate_multifactor_alpha(sample, return_column, lags=12):
    calendar = pd.period_range(sample.index.min(), sample.index.max(), freq="M")
    sample = sample.reindex(calendar)
    observed = sample[[return_column, *factor_columns]].notna().all(axis=1).to_numpy()
    outcome = sample.loc[observed, return_column].to_numpy()
    design = np.column_stack([np.ones(len(outcome)), sample.loc[observed, factor_columns]])
    coefficients, _, rank, _ = np.linalg.lstsq(design, outcome, rcond=None)
    n_obs, n_parameters = design.shape
    if n_obs <= n_parameters or rank < n_parameters:
        raise ValueError("The multifactor regression needs more observations and full-rank factors.")

    # Keep the residual score on the monthly calendar so missing returns do not compress lags.
    scores = np.zeros((len(calendar), n_parameters))
    scores[observed] = design * (outcome - design @ coefficients)[:, None]
    score_cov = scores.T @ scores
    lags = min(lags, len(calendar) - 1)
    for lag in range(1, lags + 1):
        cross_cov = scores[lag:].T @ scores[:-lag]
        score_cov += (1 - lag / (lags + 1)) * (cross_cov + cross_cov.T)
    bread = np.linalg.inv(design.T @ design)
    covariance = bread @ score_cov @ bread * n_obs / (n_obs - n_parameters)

    alpha = coefficients[0]
    alpha_se = np.sqrt(max(0.0, covariance[0, 0]))
    return {
        "months_used": n_obs,
        "alpha_monthly_pct": 100 * alpha,
        "nw_se_monthly_pct": 100 * alpha_se,
        "t_stat": alpha / alpha_se if alpha_se > 0 else np.nan,
        "market_beta": coefficients[1],
    }


monthly_returns = monthly_panel[["permno", "mthcaldt", "mthret", "mthcap", "mthdelflg"]].copy()
monthly_returns["month"] = monthly_returns.pop("mthcaldt").dt.to_period("M")
previous = monthly_returns.groupby("permno")[["month", "mthcap"]].shift()
monthly_returns["formation_cap"] = previous["mthcap"].where(previous["month"].eq(monthly_returns["month"] - 1))
size_lookup = monthly_returns.set_index(["permno", "month"])["mthcap"]
delisting_by_permno = delisting_info.set_index("permno")[["delistingdt", "delactiontype"]]

core_months = pd.period_range("1976-01", "2022-12", freq="M")
recent_months = pd.period_range("2023-01", "2025-12", freq="M")
if monthly_factors.reindex(core_months.union(recent_months)).isna().any().any():
    raise ValueError("The factor archives must cover January 1976 through December 2025.")


def build_holdings(register, holding_months, months, exclude_largest=0):
    events = register[["permno", "effective_date"]].dropna().astype({"permno": "int64"}).copy()
    events["effective_month"] = events["effective_date"].dt.to_period("M")
    if exclude_largest:
        # Size at the month-end before the effective month is known at the event date.
        size_keys = pd.MultiIndex.from_arrays([events["permno"], events["effective_month"] - 1])
        size = pd.Series(size_lookup.reindex(size_keys).to_numpy(), index=events.index)
        events = events.loc[~size.rank(ascending=False, method="first").le(exclude_largest)]
    holdings = events.merge(pd.DataFrame({"holding_month": range(1, holding_months + 1)}), how="cross")
    holdings["month"] = holdings["effective_month"] + holdings["holding_month"]
    holdings = holdings.join(delisting_by_permno, on="permno")
    delisting_month = holdings["delistingdt"].where(holdings["delistingdt"].gt(holdings["effective_date"])).dt.to_period("M")
    holdings["delists_this_month"] = holdings["month"].eq(delisting_month)
    holdings = holdings.loc[
        holdings["month"].isin(months) & (delisting_month.isna() | holdings["month"].le(delisting_month))
    ]
    return holdings.sort_values("effective_date").drop_duplicates(["permno", "month"], keep="last")


def leg_positions(holdings, weighting="equal", delisting_assumption=None):
    positions = holdings.merge(
        monthly_returns[["permno", "month", "mthret", "formation_cap", "mthdelflg"]],
        on=["permno", "month"], how="left", validate="one_to_one",
    )
    returns = positions["mthret"]
    if delisting_assumption is not None:
        partial = positions["mthdelflg"].isin(["G", "M", "V"]) & ~positions["delactiontype"].isin(["MER", "GEX"])
        returns = returns.where(~partial, (1 + returns) * (1 + delisting_assumption) - 1)
    positions["ret"] = returns
    missing = int(positions["ret"].isna().sum())
    positions = positions.dropna(subset=["ret"])
    if weighting == "value":
        positions = positions.loc[positions["formation_cap"].gt(0)].copy()
        positions["weight"] = positions["formation_cap"] / positions.groupby("month")["formation_cap"].transform("sum")
    else:
        positions["weight"] = 1 / positions.groupby("month")["permno"].transform("size")
    positions.attrs["missing_returns"] = missing
    return positions


def leg_series(positions, months):
    grouped = positions.assign(weighted=positions["weight"] * positions["ret"]).groupby("month")
    return pd.DataFrame({
        "ret": grouped["weighted"].sum(), "positions": grouped.size(),
    }).reindex(months).assign(positions=lambda frame: frame["positions"].fillna(0).astype(int))


def strategy_returns(deletion_register, addition_register, months, holding_months=12,
                     weighting="equal", delisting_assumption=None, exclude_largest=0):
    legs = {}
    for name, register in [("deletions", deletion_register), ("additions", addition_register)]:
        holdings = build_holdings(register, holding_months, months, exclude_largest)
        positions = leg_positions(holdings, weighting, delisting_assumption)
        legs[name] = {"positions": positions, "series": leg_series(positions, months),
                      "events": holdings.drop_duplicates(["permno", "effective_date"]).shape[0]}
    frame = pd.DataFrame({
        "deletions_ret": legs["deletions"]["series"]["ret"],
        "additions_ret": legs["additions"]["series"]["ret"],
        "deletions_positions": legs["deletions"]["series"]["positions"],
        "additions_positions": legs["additions"]["series"]["positions"],
    }).join(monthly_factors)
    frame.loc[frame["deletions_positions"].eq(0), "deletions_ret"] = np.nan
    frame.loc[frame["additions_positions"].eq(0), "additions_ret"] = np.nan
    frame["Long deletions"] = frame["deletions_ret"] - frame["RF"]
    frame["Short additions"] = frame["RF"] - frame["additions_ret"]
    frame["Long-short spread"] = frame["deletions_ret"] - frame["additions_ret"]
    return frame, legs


strategies = ["Long deletions", "Short additions", "Long-short spread"]


def alpha_table(frame, legs, periods=all_periods, columns=None, strategy_list=strategies):
    rows = []
    for strategy in strategy_list:
        column = (columns or {}).get(strategy, strategy)
        for period in periods:
            if period == overall_period:
                sample = frame
            elif period in sub_periods:
                sample = frame.loc[period_of(frame.index.year) == period]
            else:
                sample = frame
            estimate = estimate_multifactor_alpha(sample, column)
            leg_names = {"Long deletions": ["deletions"], "Short additions": ["additions"],
                         "Long-short spread": ["deletions", "additions"]}[strategy]
            in_period = sample.index
            events = sum(
                legs[leg]["positions"].loc[legs[leg]["positions"]["month"].isin(in_period)]
                .drop_duplicates(["permno", "effective_date"]).shape[0]
                for leg in leg_names
            )
            rows.append({"strategy": strategy, "period": period, "events": events, **estimate})
    return pd.DataFrame(rows).set_index(["strategy", "period"])


core_frame, core_legs = strategy_returns(main_deletions, main_additions, core_months)
display(pd.DataFrame({
    leg: {
        "events held": core_legs[leg]["events"],
        "months with a position": int(core_frame[f"{leg}_positions"].gt(0).sum()),
        "mean positions per month": core_frame[f"{leg}_positions"].mean(),
        "months with fewer than 5 positions": int(core_frame[f"{leg}_positions"].between(1, 4).sum()),
        "position-months without a CRSP return (dropped)": core_legs[leg]["positions"].attrs["missing_returns"],
        "position-months with a missing or partial delisting return (G, M, V)": int(
            core_legs[leg]["positions"]["mthdelflg"].isin(["G", "M", "V"]).sum()
        ),
    } for leg in ["deletions", "additions"]
}).round(2))

## Calendar-time portfolios: gross alphas

Monthly alpha in percent on the five Fama–French factors and momentum, gross of all costs, with 12 Newey–West lags. `events` counts the events held in at least one month of the period. The short leg's alpha is the negative of the additions basket's alpha.

In [ ]:
gross_alphas = alpha_table(core_frame, core_legs)
display(gross_alphas.round(3))
log_specifications("Calendar-time gross alphas, EW 12 months", gross_alphas)

delisting_frame, delisting_legs = strategy_returns(main_deletions, main_additions, core_months, delisting_assumption=-0.30)
delisting_alphas = alpha_table(delisting_frame, delisting_legs)
print("Sensitivity: missing or partial non-merger delisting returns replaced by an assumed -30%")
display(delisting_alphas[["months_used", "alpha_monthly_pct", "t_stat"]].round(3))
log_specifications("Calendar-time alphas, delisting-return assumption", delisting_alphas)

### Interpretation of the gross alphas

The long–short spread earns a gross monthly alpha of 0.67% over 1976–2022 (t = 1.65), mostly from the deletion leg. The alpha is concentrated in 1990–1999 (3.33% per month, t = 2.76) and, less precisely, 2000–2009 (1.38%, t = 1.53). It is zero in 2010–2022 (0.04%, t = 0.11) and negative in 1976–1989 (−0.51%, t = −0.51). The short additions leg contributes little in any period: its largest alpha is 0.64% in 1990–1999 (t = 2.03). The calendar-time results therefore agree with the event study. The post-event drift that a portfolio could capture was large in the 1990s and 2000s and has disappeared since 2010.

The deletion leg is thin: it holds about nine stocks on average, and 151 of its 561 months hold fewer than five, mostly in 1976–1989. Its returns are volatile and its market beta is above one. The −30% delisting assumption affects only one position-month and leaves the estimates unchanged. 78 deletion position-months (nine securities) have a CRSP row without a return, probably while the stock traded off-exchange. They are left out of that month's average, which may bias the deletion leg if those months were poor.

## Transaction and borrowing costs

**Turnover.** Each month, the cost is the per-side cost times the absolute change from the drifted weights of the previous month to the new weights. This counts entries, exits, and the rebalancing that keeps the portfolio equal weighted. A full replacement of the portfolio has a one-way turnover of 2. A position that ends with a delisting is paid out in cash and incurs no exit cost. The first month of the sample is charged for building the whole portfolio.

**Cost per side.**

- *Estimated.* Half the quoted bid–ask spread, `(ask − bid) / midpoint / 2`, from the last valid daily CRSP quote in the month before the trade. A quote is valid if `0 < bid < ask` and the relative spread is below 20%. A missing estimate is replaced by the median estimate among that month's trades. If no trade that month has a quote, an assumed 0.5% is used. Quoted spreads leave out price impact, which is likely substantial for index-event trades at the closing auction. They also overstate the effective spread for large stocks. The estimate is therefore a rough benchmark, not a precise cost.
- *Assumed.* 0.2%, 0.5%, and 1.0% per side for every trade.

**Borrowing.** The short additions leg pays an annual borrow fee of 0.5%, 1%, or 2% on the short notional. These scenarios are combined with the estimated trading costs.

Net alphas use the same regression as the gross alphas, with the cost subtracted from each month's return. The break-even cost is the per-side cost at which the long–short alpha falls to zero: gross alpha divided by the average monthly one-way turnover of both legs.

In [ ]:
quotes = daily.loc[
    daily["dlybid"].gt(0) & daily["dlyask"].gt(daily["dlybid"]), ["permno", "date", "dlybid", "dlyask"]
].copy()
quotes["relative_spread"] = (quotes["dlyask"] - quotes["dlybid"]) / ((quotes["dlyask"] + quotes["dlybid"]) / 2)
quotes = quotes.loc[quotes["relative_spread"].lt(0.20)]
quotes["quote_month"] = quotes["date"].dt.to_period("M")
month_end_half_spread = (
    quotes.sort_values("date").groupby(["permno", "quote_month"])["relative_spread"].last() / 2
).rename("half_spread")
fallback_half_spread = 0.005


def leg_trades(positions, series):
    """Absolute weight change per security at the start of each month, with the estimated half-spread."""
    current = positions[["permno", "month", "weight"]]
    previous = positions[["permno", "month", "weight", "ret", "delists_this_month"]].copy()
    previous = previous.join(series["ret"].rename("portfolio_ret"), on="month")
    previous["drifted"] = previous["weight"] * (1 + previous["ret"]) / (1 + previous["portfolio_ret"])
    previous["month"] = previous["month"] + 1
    trades = current.merge(
        previous[["permno", "month", "drifted", "delists_this_month"]], on=["permno", "month"], how="outer"
    )
    trades = trades.loc[trades["month"].isin(series.index)]
    trades["traded"] = (trades["weight"].fillna(0) - trades["drifted"].fillna(0)).abs()
    exit_by_delisting = trades["weight"].isna() & trades["delists_this_month"].eq(True)
    trades.loc[exit_by_delisting, "traded"] = 0.0
    trades["quote_month"] = trades["month"] - 1
    trades = trades.join(month_end_half_spread, on=["permno", "quote_month"])
    trades["estimated"] = trades["half_spread"].notna()
    month_median = trades.groupby("month")["half_spread"].transform("median")
    trades["half_spread"] = trades["half_spread"].fillna(month_median).fillna(fallback_half_spread)
    return trades


def leg_costs(trades, months):
    by_month = trades.assign(
        spread_cost=trades["traded"] * trades["half_spread"]
    ).groupby("month").agg(turnover=("traded", "sum"), spread_cost=("spread_cost", "sum"))
    return by_month.reindex(months).fillna(0.0)


def add_net_columns(frame, legs):
    costs = {}
    for leg in ["deletions", "additions"]:
        trades = leg_trades(legs[leg]["positions"], legs[leg]["series"])
        costs[leg] = leg_costs(trades, frame.index)
        costs[leg + "_trades"] = trades
    net = frame.copy()
    for leg in ["deletions", "additions"]:
        net[f"{leg}_turnover"] = costs[leg]["turnover"]
        net[f"{leg}_cost_estimated"] = costs[leg]["spread_cost"]
    scenarios = {"estimated spread": None, "0.2% per side": 0.002, "0.5% per side": 0.005, "1.0% per side": 0.010}
    columns = {}
    for scenario, per_side in scenarios.items():
        cost = {
            leg: net[f"{leg}_cost_estimated"] if per_side is None else net[f"{leg}_turnover"] * per_side
            for leg in ["deletions", "additions"]
        }
        net[f"Long deletions | {scenario}"] = net["Long deletions"] - cost["deletions"]
        net[f"Short additions | {scenario}"] = net["Short additions"] - cost["additions"]
        net[f"Long-short spread | {scenario}"] = net["Long-short spread"] - cost["deletions"] - cost["additions"]
        columns[scenario] = {strategy: f"{strategy} | {scenario}" for strategy in strategies}
    for fee in (0.005, 0.01, 0.02):
        scenario = f"estimated spread + {fee:.1%} borrow"
        net[f"Short additions | {scenario}"] = net["Short additions | estimated spread"] - fee / 12
        net[f"Long-short spread | {scenario}"] = net["Long-short spread | estimated spread"] - fee / 12
        net[f"Long deletions | {scenario}"] = np.nan
        columns[scenario] = {strategy: f"{strategy} | {scenario}" for strategy in strategies}
    return net, columns, costs


core_net, cost_columns, core_costs = add_net_columns(core_frame, core_legs)

trade_rows = pd.concat([core_costs["deletions_trades"], core_costs["additions_trades"]])
trade_rows = trade_rows.loc[trade_rows["traded"].gt(0)]
trade_rows["period"] = period_of(trade_rows["month"].dt.year)
display(pd.DataFrame({
    "share of traded weight with a quote": trade_rows.groupby("period").apply(
        lambda rows: np.average(rows["estimated"], weights=rows["traded"]), include_groups=False
    ),
    "weighted mean half-spread (%)": trade_rows.groupby("period").apply(
        lambda rows: 100 * np.average(rows["half_spread"], weights=rows["traded"]), include_groups=False
    ),
    "deletions: mean monthly turnover": core_net.groupby(period_of(core_net.index.year))["deletions_turnover"].mean(),
    "additions: mean monthly turnover": core_net.groupby(period_of(core_net.index.year))["additions_turnover"].mean(),
}).round(3))

In [ ]:
def compact(table):
    """Alpha (t) by period, for display."""
    cell = table.apply(lambda row: f"{row['alpha_monthly_pct']:.3f} ({row['t_stat']:.2f})", axis=1)
    return cell.unstack("period")[all_periods]


net_tables = {}
for scenario, columns in cost_columns.items():
    active = [strategy for strategy in strategies if core_net[columns[strategy]].notna().any()]
    net_tables[scenario] = alpha_table(core_net, core_legs, columns=columns, strategy_list=active)
    log_specifications(f"Net alphas, {scenario}", net_tables[scenario])
net_alphas = pd.concat(net_tables, names=["cost scenario"])
print("Monthly alpha in percent (Newey–West t-statistic), net of costs. The gross figures are above.")
display(compact(net_alphas))

turnover = core_net["deletions_turnover"] + core_net["additions_turnover"]
spread_alpha = gross_alphas.xs("Long-short spread", level="strategy")["alpha_monthly_pct"] / 100
break_even = pd.Series({
    period: spread_alpha[period] / (
        turnover if period == overall_period else turnover.loc[period_of(turnover.index.year) == period]
    ).loc[core_net["Long-short spread"].notna()].mean()
    for period in all_periods
}, name="break-even cost per side (%)") * 100
display(pd.DataFrame({
    "gross long-short alpha (%)": 100 * spread_alpha,
    "mean monthly one-way turnover, both legs": [
        (turnover if period == overall_period else turnover.loc[period_of(turnover.index.year) == period])
        .loc[core_net["Long-short spread"].notna()].mean() for period in all_periods
    ],
    "break-even cost per side (%)": break_even,
}).round(3))

### Interpretation of the net alphas

Estimated quoted half-spreads fall from about 1% per side for the traded weight in 1990–1999 (deletions are often small, illiquid stocks) to 0.04% in 2010–2022. Quotes cover only 15% of traded weight in 1976–1989, so costs there rely mostly on the fallback. The portfolios turn over about 0.6 of their value per month across both legs.

Over 1976–2022, the long–short alpha net of estimated spreads is 0.35% per month (t = 0.87). With a 2% borrow fee it is 0.19% (t = 0.46). With 1% per side it is 0.08% (t = 0.19). In 1990–1999 the spread survives every cost scenario: the net alpha is 2.5–3.2% per month with t between 2.08 and 2.65. In 2010–2022 the net alpha is between −0.50% and +0.02% per month under every scenario, and the break-even cost is only 0.08% per side. In 2010–2022 the strategy is not profitable even before costs, and any plausible cost makes it negative. The short additions leg alone is never profitable net of costs in a statistically meaningful way (the largest t is 1.86, in 1990–1999). In 1976–1989 it loses money: its net alpha is −0.46% to −0.73% per month, with t between −1.91 and −3.03.

Quoted spreads understate the cost of trading at index-event closes, where price impact is large, and the borrow fees are assumptions. The net alphas are therefore likely upper bounds.

## Robustness of the calendar-time results

Each check changes one choice relative to the main specification (equal weighted, 12-month holding, all main-sample events, gross returns):

1. **Value weighted**, using market capitalisation at the end of the previous month.
2. **Holding periods of 6 and 24 months.**
3. **Five largest events dropped** from each leg. Size is market capitalisation at the end of the month before the effective month, so the screen uses only information available at the event date.
4. **2023–2025 (provisional).** Return months from January 2023 to December 2025. These portfolios hold core events whose holding period extends past 2022 together with the provisional 2023–2025 events.

In [ ]:
robustness_specs = {
    "Value weighted": dict(weighting="value"),
    "Holding 6 months": dict(holding_months=6),
    "Holding 24 months": dict(holding_months=24),
    "Five largest events dropped": dict(exclude_largest=5),
}
robustness_tables = {}
for label, options in robustness_specs.items():
    frame, legs = strategy_returns(main_deletions, main_additions, core_months, **options)
    robustness_tables[label] = alpha_table(frame, legs)
    log_specifications(f"Robustness, {label}", robustness_tables[label])

recent_frame, recent_legs = strategy_returns(
    pd.concat([main_deletions, recent_deletions_sample]), pd.concat([main_additions, recent_additions_sample]),
    recent_months,
)
robustness_tables["2023–2025 (provisional)"] = alpha_table(recent_frame, recent_legs, periods=[recent_label])
log_specifications("Robustness, 2023–2025 provisional", robustness_tables["2023–2025 (provisional)"])

core_robustness = {label: table for label, table in robustness_tables.items() if label != "2023–2025 (provisional)"}
robustness_alphas = pd.concat(
    {"Main: equal weighted, 12 months": gross_alphas, "Delisting return -30%": delisting_alphas, **core_robustness},
    names=["specification"],
)
print("Monthly alpha in percent (Newey–West t-statistic), gross of costs, 1976–2022")
display(compact(robustness_alphas))
print("Robustness check: 2023–2025 (provisional membership), gross of costs")
display(robustness_tables["2023–2025 (provisional)"][["events", "months_used", "alpha_monthly_pct", "t_stat", "market_beta"]].round(3))

### Interpretation of the robustness checks

The pattern holds in every check. The 1990–1999 spread alpha is positive under every specification (1.0–4.4% per month, t between 1.30 and 2.76). The smallest value is for the 24-month holding period. The 2010–2022 spread alpha lies between −0.25% and +0.06% per month with |t| ≤ 0.50. Value weighting roughly halves the alphas in 1990–2009, so smaller deletions drive more of the effect. A 6-month holding period gives larger monthly alphas than 12 months, and a 24-month period much smaller ones, which is consistent with a rebound concentrated in the first months after deletion. Dropping the five largest events changes little. In the provisional 2023–2025 period, the spread alpha is −0.42% per month (t = −0.48), with long deletions at −1.38% (t = −1.55) and short additions at +0.95% (t = 1.45), based on 36 months.

## Post hoc check: spin-offs removed shortly after their addition

Inspection of the daily data after the main results were computed showed 19 main-sample deletions, all from 2015–2022, whose securities had no trading history before the deletion. These are spin-offs that S&P adds on the distribution date and removes a few days later. They are the deletion counterpart of the spin-off additions that the main sample already excludes, and the flag `flag_no_prior_crsp_3m` uses only information available at the event date. Because the check was defined after the results were seen, the main sample is left unchanged. The check below drops these events from the deletion sample and reruns the event study and the deletion-based portfolios.

In [ ]:
spin_free_deletions = main_deletions.loc[~main_deletions["flag_no_prior_crsp_3m"]]
print(f"Deletions dropped: {len(main_deletions) - len(spin_free_deletions)}")
post_hoc_cars = summarize_cars(compute_cars(study_events(spin_free_deletions), "index_trade_date"))
print("Deletions without spin-off removals: mean CAR (%), clustered t-statistic")
display(post_hoc_cars[["events", "mean_car_pct", "t_stat", "median_car_pct"]].round(3))
log_specifications("Post hoc, deletion CARs without spin-off removals", post_hoc_cars)

post_hoc_frame, post_hoc_legs = strategy_returns(spin_free_deletions, main_additions, core_months)
post_hoc_alphas = alpha_table(post_hoc_frame, post_hoc_legs, strategy_list=["Long deletions", "Long-short spread"])
print("Gross alphas without spin-off removals: monthly alpha in percent (Newey–West t-statistic)")
display(compact(post_hoc_alphas))
log_specifications("Post hoc, alphas without spin-off removals", post_hoc_alphas)

Removing the 19 spin-off removals changes only 2010–2022, where all of them fall. The deletion rebound becomes slightly negative (−0.8% over [+1, +60], −1.5% over [+1, +250]), and the spread alpha becomes −0.13% per month (t = −0.34). The conclusion for 2010–2022 does not change.

## Number of specifications

The table counts every estimate reported in this notebook: each mean CAR, each difference test, and each alpha. All of them were fixed before any result was computed, and none was dropped after the results were seen. With this many tests, a few |t| > 2 are expected by chance even if no effect exists. A single estimate should be judged against that count. The more demanding threshold of |t| > 3 is a useful benchmark.

In [ ]:
specification_count = pd.DataFrame(specification_log).set_index("table")
specification_count.loc["Total"] = specification_count.sum()
display(specification_count)
headline_tables = ["Event study, main", "Event study, 2010–2022 vs 1990–1999 difference",
                   "Calendar-time gross alphas, EW 12 months"]
print("Headline tables (main event study, difference tests, gross alphas): "
      f"{specification_count.loc[headline_tables, 'estimates'].sum()} estimates")

## Summary

**Has the effect disappeared?** Partly. Additions still earn a large market-adjusted return before the index trading date, about 9% over 60 sessions in 2010–2022. This run-up cannot be traded without knowing which stocks will be added, and without announcement dates it cannot be split into announcement, anticipation, and selection effects. The part that follows the index trading date has disappeared. The post-event reversal of additions (1990s) and the strong post-event rebound of deletions (1990–2009) are both absent in 2010–2022. The change in the deletion rebound from the 1990s to 2010–2022 is statistically significant (t = −2.6 for [+1, +60] and −2.2 for [+1, +250]).

**Is there a tradable version after costs?** Not in the recent period. The long-deletions, short-additions portfolio earned large alphas in 1990–1999 that survive the cost assumptions used here. Its 2010–2022 alpha is zero before costs and negative after. Over the full 1976–2022 sample, the net alpha is positive but statistically indistinguishable from zero (0.35% per month, t = 0.87, with estimated spreads).

**Multiple testing.** The notebook reports 297 estimates: 89 have |t| > 2 and 39 have |t| > 3 (see the table above). Of the 39, 38 are event-study means, mostly for the pre-event windows and for deletions in 1990–2009. The other is the net alpha of short additions in 1976–1989 with a 2% borrow fee, which is negative. Every positive portfolio alpha with t > 2 is in 1990–1999. No portfolio alpha has t > 3. The conclusion that the post-event effect has disappeared since 2010 rests on estimates close to zero in every specification, so it does not depend on any single test.